# What's new in CDA

*Standalone -- no prior vignette required.*

**DRAFT STATUS: every count/value below is a placeholder. This notebook must be run live against the real API before recording the video, and all `[PLACEHOLDER]` markers replaced with real output. The video embed below is also a placeholder -- swap in the real embed URL once the video is recorded and uploaded.**

<iframe width="560" height="315" src="https://www.youtube.com/embed/[PLACEHOLDER_VIDEO_ID]" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

Watch the video above for the quick walkthrough, or follow along below to run everything yourself -- same search, same results, fully interactive.

You've seen our other videos on searching CDA. We just shipped something new, and this notebook walks through it with a fresh search term instead of our usual kidney example -- let's try melanoma.

In [ ]:
from cdapython import *
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt
opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0
opt.columnDefs = [{"className": "dt-left", "targets": "_all"}]

## Act 1: Bare-term global search

One word, no setup, searched across every column at once:

In [ ]:
summarize_subjects('melanoma')

`1,565 subjects`, with `78,263` related files -- from one word, zero setup. And look at the data_source breakdown: these subjects turn up across GDC, IDC, GC, CTDC, and ICDC, in every combination -- one search, reaching all the way across the data centers CDA covers.

There's already a surprise sitting in the default columns, before we even add anything: this is a search for a *human* cancer, but the species breakdown shows `28 dogs` and `20 mice` right alongside `1,488 humans`. CDA's Integrated Canine Data Commons tracks melanoma in dogs too, specifically because it helps researchers studying the human disease -- so a plain word search surfaces cross-species data you might not have thought to ask for.

Now let's add a few columns so we can see what actually matched on the human side:

In [ ]:
summarize_subjects('melanoma', add_columns=['anatomic_site', 'diagnosis', 'format'])

`[PLACEHOLDER: describe real anatomic_site breakdown once run]`

Melanoma is mostly a skin cancer, so "skin" showing up as a top value isn't a surprise. What's worth pointing out on camera is whichever **unrelated** field shows up with a large count anyway -- the same mechanism as the "blood" surprise in our kidney vignette. A matched subject brings *all* their data along, including things that have nothing to do with why they matched. `[PLACEHOLDER: identify and name the actual surprising field/value once this cell is run -- do not force a field into this slot that isn't actually surprising in the real output]`.

Separately, watch for more *specific* anatomic_site terms than "melanoma" itself rolling up automatically -- e.g. specific skin sub-sites. `[PLACEHOLDER: list the real more-specific terms that appear]`. That's CDA's ontology-backed rollup (UBERON) doing its job, the same way "right kidney" and "left kidney" rolled up under "kidney" in our other vignette.

## Act 2: The new thing -- seeing *why* something matched

Until recently, CDA could tell you *that* a result matched through ontology rollup, but not *which* term did it. Now it can, via `add_extras`:

In [ ]:
summarize_subjects('melanoma', add_columns=['anatomic_site', 'diagnosis', 'format'], add_extras=['all'])

Same subjects, same count -- `add_extras` doesn't change what matched, only annotates *why*. Look at `uvea`: its `slim_terms` value is literally `eye` -- that's the parent/child link made explicit, confirming what we guessed from the matching counts a minute ago. `blood`'s synonym_terms show `portion of blood` and `vertebrate blood`, with containing_terms like `hematopoietic system` and `bodily fluid` -- real vocabulary explaining what that match actually means.

But the diagnosis table is the cleaner demo. Every melanoma subtype here -- Nodular, Spindle cell, Epithelioid cell, Amelanotic, Lentigo maligna, even "Malignant melanoma" itself -- shares the exact same `slim_terms` value: **Nevi and melanomas**. One shared label explaining why a dozen different-sounding diagnoses all count as the same broad category. That's not a coincidence of formatting -- it's CDA showing its actual classification logic, in writing, for the first time.

Worth saying plainly: this only shows something new on two fields right now -- `anatomic_site` and `disease` -- since those are the two backed by a real external ontology (UBERON and ICD-O-3). Fields like `sex` or `species` are fully standardized but flat, with no "more specific" version to roll up to. A few fields, like `stage`, aren't harmonized yet at all -- that's not a bug, just where the work currently stands.

## Act 3: From poking around to browsing deliberately

That's the "just see what's out there" mode. Now suppose this is for something real -- a grant proposal, an actual analysis -- and we want to know what we're working with before committing to a filter.

First, the big picture -- what's even searchable:

In [ ]:
tables()

I've got melanoma subjects and files -- now say I actually want to *do* something with this. I'm interested in sequencing data specifically, not imaging or clinical records. Rather than guess at a column name, search by what a column is described as doing:

In [ ]:
columns(description='format')

`format` looks right. Before filtering on it, let's see what the real values look like:

In [ ]:
column_values('format', data_source='GDC')

VCF -- variant call format, a real sequencing output -- is already one of the largest values across our melanoma files, with over 13,000 of them. Let's see how many human melanoma subjects actually have VCF data:

In [ ]:
summarize_subjects(match_all=['diagnosis = *melanoma*', 'format = VCF', 'species = human'])

`[PLACEHOLDER: X subjects]` -- a focused, real number of human melanoma subjects with sequencing data ready to use, reached by checking before filtering instead of guessing.

## Wrap-up

Two searches -- one bare word, one careful filter -- plus a brand-new way to see exactly why CDA matched what it matched. No install required to try this yourself; see Getting Started. For the full walkthroughs this notebook draws from, see [Is there anything about X?](01_global_search.ipynb) and [Getting oriented before I filter](02_systematic_browsing.ipynb).